# Modular Neuro-Symbolic News Pipeline


## 1. Azure Grok API setup

Run this installation cell once in the notebook environment if needed:

```python
%pip install -U openai azure-identity python-dotenv
```

The notebook uses the Azure AI Foundry OpenAI-compatible endpoint for the deployed
`grok-4-1-fast-reasoning` model. Add `GROK_TARGET_URI` and `AZURE_TENANT_ID` to `.env`; the
notebook authenticates through a browser-based Azure sign-in and never prints credentials.

The saved defaults are mock-only: `USE_MOCK_LLM=True` and
`ALLOW_LIVE_API_CALLS=False`, so the notebook cannot spend API credits until both are
changed deliberately.

In [26]:
from __future__ import annotations

import hashlib
import json
import os
import re
import time
from collections import deque
from datetime import datetime
from pathlib import Path
from typing import Literal

import pandas as pd
import torch
from azure.identity import InteractiveBrowserCredential, get_bearer_token_provider
from dotenv import load_dotenv
from huggingface_hub import hf_hub_download
from openai import OpenAI
from pydantic import BaseModel, Field, ValidationError

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_columns", 50)

# Search from the current working directory. In the local dissertation workspace,
# also try the sibling Sentiment project without exposing any secret values.

PROJECT_ROOT = Path("/home/jovyan/Stock-Sentiment-Prediction")
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

dotenv_candidates = [
    PROJECT_ROOT / "env.txt",
    PROJECT_ROOT / ".env",
    *[
        parent / "Sentiment project" / "env.txt"
        for parent in (Path.cwd(), *Path.cwd().parents)
    ],
    *[
        parent / "Sentiment project" / ".env"
        for parent in (Path.cwd(), *Path.cwd().parents)
    ],
]
dotenv_path = next((path for path in dotenv_candidates if path.exists()), None)

if dotenv_path:
    load_dotenv(dotenv_path=str(dotenv_path))

HF_TOKEN = os.getenv("HF_TOKEN")
assert HF_TOKEN, "HF_TOKEN is missing. Add it to your env.txt or .env file before continuing."

AZURE_ENDPOINT = os.getenv("GROK_TARGET_URI")
AZURE_TENANT_ID = os.getenv("AZURE_TENANT_ID")
assert AZURE_ENDPOINT, "GROK_TARGET_URI is missing. Add the full Azure OpenAI /openai/v1 URL to .env."
assert AZURE_TENANT_ID, "AZURE_TENANT_ID is missing. Add the Microsoft Entra tenant GUID to .env."

# This local notebook uses an explicit Microsoft Entra browser sign-in. The first
# live request opens a browser; its token is cached for the current session.
token_provider = get_bearer_token_provider(
    InteractiveBrowserCredential(tenant_id=AZURE_TENANT_ID),
    "https://ai.azure.com/.default",
)
client = OpenAI(base_url=AZURE_ENDPOINT, api_key=token_provider)

DATASET_REPO = "cookekieran/alphavantage-market-news"
MODEL_ID = os.getenv("GROK_DEPLOYMENT_NAME", "grok-4-1-fast-reasoning")
MONTH = "2023-01"
TICKER = "AAPL"
MIN_TICKER_RELEVANCE = 0.20
MAX_ARTICLES = 1

# Keep True while learning the pipeline. It creates deterministic placeholder
# assessments so every non-model stage can be inspected cheaply.
USE_MOCK_LLM = False
ALLOW_LIVE_API_CALLS = True

# Set to 1 while inspecting individual article behaviour. Increase later for throughput.
GROK_BATCH_SIZE = int(os.getenv("GROK_BATCH_SIZE", "1"))
GROK_SLEEP_SECONDS = float(os.getenv("GROK_SLEEP_SECONDS", "0"))
MAX_COMPLETION_TOKENS = int(os.getenv("GROK_MAX_COMPLETION_TOKENS", "200"))
ARTICLE_TITLE_CHARS = 180
ARTICLE_SUMMARY_CHARS = 600
PROMPT_EVENT_LIMIT = 3

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "modular_news_demo" / f"{TICKER}_{MONTH}_{MODEL_ID.replace('/', '_')}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print({
    "dataset": DATASET_REPO,
    "model": MODEL_ID,
    "month": MONTH,
    "ticker": TICKER,
    "max_articles": MAX_ARTICLES,
    "use_mock_llm": USE_MOCK_LLM,
    "grok_batch_size": GROK_BATCH_SIZE,
    "live_calls_allowed": ALLOW_LIVE_API_CALLS,
    "max_completion_tokens": MAX_COMPLETION_TOKENS,
    "article_summary_chars": ARTICLE_SUMMARY_CHARS,
    "prompt_event_limit": PROMPT_EVENT_LIMIT,
    "output_dir": str(OUTPUT_DIR),
})


{'dataset': 'cookekieran/alphavantage-market-news', 'model': 'grok-4-1-fast-reasoning', 'month': '2023-01', 'ticker': 'AAPL', 'max_articles': 1, 'use_mock_llm': False, 'grok_batch_size': 1, 'live_calls_allowed': True, 'max_completion_tokens': 200, 'article_summary_chars': 600, 'prompt_event_limit': 3, 'output_dir': 'c:\\Users\\kiera\\OneDrive\\University\\Univeristy\\Studying\\Masters\\Dissertation\\Sentiment project\\outputs\\modular_news_demo\\AAPL_2023-01_grok-4-1-fast-reasoning'}


## 2. Download and join the January news partitions

The dataset stores:

- `articles`: title, summary, source, timestamp, and stable `article_uid`;
- `tickers`: ticker relevance and Alpha Vantage ticker sentiment;
- `topics`: topic relevance.

We filter through the ticker-link table rather than relying on
`requested_entity`, because an article requested for one entity may still be
relevant to another.

In [27]:
def download_partition(folder: str) -> Path:
    return Path(
        hf_hub_download(
            repo_id=DATASET_REPO,
            repo_type="dataset",
            filename=f"{folder}/{MONTH}.parquet",
            token=HF_TOKEN,
        )
    )


articles = pd.read_parquet(download_partition("articles"))
tickers = pd.read_parquet(download_partition("tickers"))
topics = pd.read_parquet(download_partition("topics"))

print("articles:", articles.shape)
print("tickers:", tickers.shape)
print("topics:", topics.shape)
display(articles.head(2))

articles: (2344, 11)
tickers: (4220, 7)
topics: (6254, 5)


,article_id,article_uid,time_published,overall_sentiment_score,overall_sentiment_label,title,summary,source,url,requested_entity,month
0,0,cc943125f2c299c334e9484c073748f4,2023-01-02 04:52:00,0.269256,Somewhat-Bullish,What is TSMC?,"Taiwan Semiconductor Manufacturing Co (TSMC) is the world's largest semiconductor manufacturer, producing chips for major tech companies...",Tech Monitor,https://www.techmonitor.ai/what-is/what-is-tsmc/,AAPL,2023-01
1,1,58d72b86288d4fffa86b70eac0d65e89,2023-01-03 03:59:02,-0.175983,Somewhat-Bearish,Apple’s market value drops below $2 trillion,"Apple Inc.'s market capitalization fell below $2 trillion for the first time since March 2021, driven by concerns over high inflation an...",Al Jazeera,https://www.aljazeera.com/economy/2023/1/3/apples-market-value-drops-below-2-trillion,MSFT,2023-01


In [28]:
ticker_links = (
    tickers.loc[
        (tickers["ticker"] == TICKER)
        & (tickers["relevance_score"] >= MIN_TICKER_RELEVANCE)
    ]
    .rename(
        columns={
            "relevance_score": "ticker_relevance",
            "ticker_sentiment_score": "av_ticker_sentiment_score",
            "ticker_sentiment_label": "av_ticker_sentiment_label",
        }
    )
    [
        [
            "article_uid",
            "ticker",
            "ticker_relevance",
            "av_ticker_sentiment_score",
            "av_ticker_sentiment_label",
        ]
    ]
)

topic_lists = (
    topics.sort_values(["article_uid", "relevance_score"], ascending=[True, False])
    .groupby("article_uid")
    .apply(
        lambda frame: [
            {"topic": row.topic, "relevance": float(row.relevance_score)}
            for row in frame.itertuples()
        ],
        include_groups=False,
    )
    .rename("topics")
    .reset_index()
)

article_stream = (
    articles.merge(ticker_links, on="article_uid", how="inner", validate="one_to_one")
    .merge(topic_lists, on="article_uid", how="left", validate="one_to_one")
    .drop_duplicates("article_uid")
    .sort_values(["time_published", "article_uid"])
    .head(MAX_ARTICLES)
    .reset_index(drop=True)
)
article_stream["topics"] = article_stream["topics"].apply(
    lambda value: value if isinstance(value, list) else []
)

print(f"Selected {len(article_stream)} chronological {TICKER} articles.")
display(
    article_stream[
        [
            "time_published",
            "title",
            "source",
            "ticker_relevance",
            "av_ticker_sentiment_label",
        ]
    ]
)

Selected 1 chronological AAPL articles.


,time_published,title,source,ticker_relevance,av_ticker_sentiment_label
0,2023-01-02 04:52:00,What is TSMC?,Tech Monitor,0.718922,Somewhat-Bullish


## 3. Define the LLM's structured output

The LLM returns **ordinal event attributes**, not unexplained decimal scores.
Every category has a stable definition in the prompt.

Event types and transmission channels are intentionally broader than individual
historical events. For example, an article about a specific conflict may create
an event called `Taiwan semiconductor conflict risk`, while its reusable
attributes include:

```text
event_type = geopolitical_conflict
transmission_channels = [supply_chain, operational_disruption]
```

Later, those attributes ground stable predicates such as
`GeopoliticalRisk(AAPL, date)` and `SupplyChainRisk(AAPL, date)`.

In [29]:
Level = Literal["none", "low", "medium", "high", "very_high"]
Direction = Literal["risk_on", "risk_off", "mixed", "neutral"]
Scope = Literal["ticker", "sector", "macro", "broad_market", "irrelevant"]
EventAction = Literal[
    "new",
    "confirm",
    "escalate",
    "deescalate",
    "contradict",
    "repeat",
    "resolve",
    "irrelevant",
]
EventType = Literal[
    "earnings_outlook",
    "product_or_innovation",
    "demand_change",
    "supply_chain",
    "regulatory_or_legal",
    "geopolitical_conflict",
    "management_or_strategy",
    "financing_or_balance_sheet",
    "competitive_position",
    "operational_disruption",
    "macroeconomic",
    "market_sentiment",
    "other",
]
TransmissionChannel = Literal[
    "revenue_growth",
    "demand_weakening",
    "margin_pressure",
    "input_cost_pressure",
    "financing_conditions",
    "supply_chain",
    "operational_disruption",
    "regulatory_restriction",
    "competitive_position",
    "market_uncertainty",
    "none",
]


class ArticleEventAssessment(BaseModel):
    article_uid: str
    ticker: str
    relevant: bool
    matched_event_id: str | None = None
    event_action: EventAction
    event_name: str
    event_type: EventType
    direction: Direction
    scope: Scope
    transmission_channels: list[TransmissionChannel]
    materiality: Level
    novelty: Level
    confidence: Level


LEVEL_VALUE = {
    "none": 0.0,
    "low": 0.25,
    "medium": 0.50,
    "high": 0.75,
    "very_high": 1.0,
}


def model_schema(model_class: type[BaseModel]) -> dict:
    # Support both Pydantic v1 and v2.
    if hasattr(model_class, "model_json_schema"):
        return model_class.model_json_schema()
    return model_class.schema()


def model_dump_compat(model: BaseModel) -> dict:
    # Support both Pydantic v1 and v2.
    if hasattr(model, "model_dump"):
        return model.model_dump()
    return model.dict()


def model_validate_json_compat(model_class: type[BaseModel], payload: str) -> BaseModel:
    # Support both Pydantic v1 and v2.
    if hasattr(model_class, "model_validate_json"):
        return model_class.model_validate_json(payload)
    return model_class.parse_raw(payload)

## 4. Temporal event-memory tables

The LLM has no persistent memory between calls. Before each article, Python
supplies a compact context packet containing:

- candidate active events created by earlier articles;
- recent article assessments;
- the current article.

After validation, Python updates three auditable temporal tables:

1. `events`: current event records;
2. `event_state_history`: every historical state change;
3. `event_evidence`: which article caused each change.

Because articles are processed chronologically, a historical call cannot see a
future event update.

In [30]:
def stable_event_id(ticker: str, event_type: str, event_name: str) -> str:
    digest = hashlib.sha1(event_name.lower().encode("utf-8")).hexdigest()[:10]
    return f"{ticker.lower()}__{event_type}__{digest}"

def max_level(a: str, b: str) -> str:
    return max([a, b], key=lambda level: LEVEL_VALUE[level])


def bump_confidence(previous: str, new: str, action: str) -> str:
    previous_value = LEVEL_VALUE.get(previous, 0.0)
    new_value = LEVEL_VALUE.get(new, 0.0)

    if action == "repeat":
        # Repetition gives only a small confidence bump.
        combined = min(1.0, max(previous_value, new_value) + 0.05)
    elif action == "confirm":
        # New supporting evidence gives a stronger bump.
        combined = min(1.0, max(previous_value, new_value) + 0.15)
    else:
        combined = max(previous_value, new_value)

    return min(LEVEL_VALUE, key=lambda level: abs(LEVEL_VALUE[level] - combined))


class TemporalEventMemory:
    def __init__(self):
        self.events: dict[str, dict] = {}
        self.state_history: list[dict] = []
        self.evidence: list[dict] = []
        self.recent_assessments = deque(maxlen=0)

    def candidate_events(self, ticker: str, limit: int = PROMPT_EVENT_LIMIT) -> list[dict]:
        active = [
            event
            for event in self.events.values()
            if event["status"] != "resolved"
            and (event["ticker"] == ticker or event["scope"] in {"sector", "macro", "broad_market"})
        ]
        active.sort(key=lambda event: event["last_updated"], reverse=True)
        return active[:limit]

    def context_packet(self, article: pd.Series) -> dict:
        """Only send fields that the model needs to classify and match an event."""
        candidates = self.candidate_events(article.ticker)
        return {
            "ticker": article.ticker,
            "article": {
                "article_uid": article.article_uid,
                "title": str(article.title)[:ARTICLE_TITLE_CHARS],
                "summary": str(article.summary)[:ARTICLE_SUMMARY_CHARS],
            },
            "candidate_active_events": [
                {
                    "event_id": event["event_id"],
                    "event_name": event["event_name"],
                    "event_type": event["event_type"],
                    "scope": event["scope"],
                    "direction": event["direction"],
                    "transmission_channels": event["transmission_channels"],
                    "status": event["status"],
                }
                for event in candidates
            ],
        }
    def apply(self, assessment: ArticleEventAssessment, observed_at: pd.Timestamp) -> str | None:
        if not assessment.relevant or assessment.event_action == "irrelevant":
            self.recent_assessments.append(model_dump_compat(assessment))
            return None

        event_id = assessment.matched_event_id
        if event_id not in self.events:
            event_id = stable_event_id(
                assessment.ticker,
                assessment.event_type,
                assessment.event_name,
            )

        previous = self.events.get(event_id, {})
        status = {
            "new": "unresolved",
            "confirm": previous.get("status", "unresolved"),
            "repeat": previous.get("status", "unresolved"),
            "escalate": "escalating",
            "deescalate": "deescalating",
            "contradict": "contested",
            "resolve": "resolved",
        }[assessment.event_action]

        event = {
            "event_id": event_id,
            "ticker": assessment.ticker,
            "event_name": assessment.event_name,
            "event_type": assessment.event_type,
            "scope": assessment.scope,
            "direction": assessment.direction,
            "transmission_channels": assessment.transmission_channels,
            
            "materiality": max_level(
                previous.get("materiality", assessment.materiality),
                assessment.materiality),
            
            "confidence": bump_confidence(
                previous.get("confidence", assessment.confidence),
                assessment.confidence,
                assessment.event_action),
            
            "status": status,
            "first_seen": previous.get("first_seen", observed_at.isoformat()),
            "last_updated": observed_at.isoformat(),
        }
        self.events[event_id] = event
        self.state_history.append(
            {
                **event,
                "valid_from": observed_at.isoformat(),
                "event_action": assessment.event_action,
            }
        )
        self.evidence.append(
            {
                "event_id": event_id,
                "article_uid": assessment.article_uid,
                "observed_at": observed_at.isoformat(),
                "event_action": assessment.event_action,
            }
        )
        recent = model_dump_compat(assessment)
        recent["resolved_event_id"] = event_id
        self.recent_assessments.append(recent)
        return event_id

    def tables(self) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        return (
            pd.DataFrame(self.events.values()),
            pd.DataFrame(self.state_history),
            pd.DataFrame(self.evidence),
        )

In [31]:
# Inspect exactly what the first model call can see.
preview_memory = TemporalEventMemory()
first_context_packet = preview_memory.context_packet(article_stream.iloc[0])
print(json.dumps(first_context_packet, indent=2, default=str)[:5000])

{
  "ticker": "AAPL",
  "article": {
    "article_uid": "cc943125f2c299c334e9484c073748f4",
    "title": "What is TSMC?",
    "summary": "Taiwan Semiconductor Manufacturing Co (TSMC) is the world's largest semiconductor manufacturer, producing chips for major tech companies like Apple, Amazon, and Google. It focuses solely on manufacturing for third parties, unlike competitors, and boasts significant market dominance and advanced process technology. The article also discusses the potential impact of a Chinese invasion of Taiwan on TSMC and its global operations, as well as the company's expansion plans in the US."
  },
  "candidate_active_events": []
}


## 5. Grok prompt and structured-response parser

Grok receives all instructions in the user prompt. The prompt explicitly
prohibits regime prediction and asks for a final JSON object between markers.

The raw response is retained before parsing. This allows malformed outputs to
be audited later. `event_summary` is deliberately omitted: the raw article is
already traceable through `article_uid`, while `event_name` retains the information needed for matching and audit.

In [32]:
ALLOWED_EVENT_TYPES = {
    "earnings_outlook", "product_or_innovation", "demand_change", "supply_chain",
    "regulatory_or_legal", "geopolitical_conflict", "management_or_strategy",
    "financing_or_balance_sheet", "competitive_position", "operational_disruption",
    "macroeconomic", "market_sentiment", "other",
}
ALLOWED_ACTIONS = {"new", "confirm", "escalate", "deescalate", "contradict", "repeat", "resolve", "irrelevant"}
ALLOWED_SCOPES = {"ticker", "sector", "macro", "broad_market", "irrelevant"}
ALLOWED_DIRECTIONS = {"risk_on", "risk_off", "mixed", "neutral"}
ALLOWED_CHANNELS = {"revenue_growth", "demand_weakening", "margin_pressure", "input_cost_pressure", "financing_conditions", "supply_chain", "operational_disruption", "regulatory_restriction", "competitive_position", "market_uncertainty", "none"}
ALLOWED_LEVELS = set(LEVEL_VALUE)

# Compact JSON aliases reduce both prompt and completion tokens.
KEY_ALIASES = {"r": "relevant", "i": "matched_event_id", "a": "event_action", "n": "event_name", "y": "event_type", "d": "direction", "s": "scope", "c": "transmission_channels", "m": "materiality", "v": "novelty", "f": "confidence"}
VALUE_ALIASES = {
    "event_type": {"earn": "earnings_outlook", "prod": "product_or_innovation", "demand": "demand_change", "supply": "supply_chain", "legal": "regulatory_or_legal", "geo": "geopolitical_conflict", "mgmt": "management_or_strategy", "finance": "financing_or_balance_sheet", "comp": "competitive_position", "ops": "operational_disruption", "macro": "macroeconomic", "market": "market_sentiment"},
    "event_action": {"con": "confirm", "esc": "escalate", "deesc": "deescalate", "contra": "contradict", "irr": "irrelevant"},
    "direction": {"on": "risk_on", "off": "risk_off", "mix": "mixed"},
    "scope": {"mkt": "broad_market", "irr": "irrelevant"},
    "materiality": {"n": "none", "l": "low", "m": "medium", "h": "high", "vh": "very_high"},
    "novelty": {"n": "none", "l": "low", "m": "medium", "h": "high", "vh": "very_high"},
    "confidence": {"n": "none", "l": "low", "m": "medium", "h": "high", "vh": "very_high"},
    "transmission_channels": {"rev": "revenue_growth", "dem": "demand_weakening", "margin": "margin_pressure", "cost": "input_cost_pressure", "fin": "financing_conditions", "supply": "supply_chain", "ops": "operational_disruption", "reg": "regulatory_restriction", "comp": "competitive_position", "unc": "market_uncertainty"},
}

COMPACT_RULES = """Classify only this article for the target ticker. No price/regime prediction. r=false unless it states a direct effect on the ticker's operations, revenue, costs, strategy, financing, competition, or regulation; ETFs, lists, peers, background, generic market commentary, recommendations and reviews are irrelevant. If relevant, match an existing event only when actors, development and mechanism are the same; otherwise new. Use no future knowledge. Return minified JSON only, no reasoning."""


def build_prompt(context_packet: dict) -> str:
    return f'''{COMPACT_RULES}
Keys: r(bool),i(candidate event_id|null),a(action),n(name <=50 chars),y(type),d(direction),s(scope),c(channels),m(materiality),v(novelty),f(confidence).
If irrelevant output {{"r":false}}. If new, i=null.
a: new|con|esc|deesc|contra|repeat|resolve|irr.
y: earn|prod|demand|supply|legal|geo|mgmt|finance|comp|ops|macro|market|other.
d: on|off|mix|neutral. s: ticker|sector|macro|mkt. c: rev|dem|margin|cost|fin|supply|ops|reg|comp|unc|none. m/v/f: n|l|m|h|vh.
P={json.dumps(context_packet, separators=(",", ":"), default=str)}'''


def normalise_enum(value: object) -> str:
    return str(value).strip().lower().replace("-", "_").replace(" ", "_")


def normalise_bool(value: object) -> bool:
    if isinstance(value, bool):
        return value
    return normalise_enum(value) in {"true", "1", "yes"}


def model_dump_compat(model: BaseModel) -> dict:
    return model.model_dump() if hasattr(model, "model_dump") else model.dict()


def model_validate_json_compat(model_class: type[BaseModel], payload: str) -> BaseModel:
    return model_class.model_validate_json(payload) if hasattr(model_class, "model_validate_json") else model_class.parse_raw(payload)


def sanitise_assessment_payload(data: dict, context_packet: dict) -> dict:
    repaired = {KEY_ALIASES.get(key, key): value for key, value in dict(data).items()}
    article = context_packet["article"]
    candidate_ids = {event["event_id"] for event in context_packet["candidate_active_events"]}
    repaired["article_uid"], repaired["ticker"] = article["article_uid"], context_packet["ticker"]
    repaired["relevant"] = normalise_bool(repaired.get("relevant", False))

    for field, allowed, default in [
        ("event_type", ALLOWED_EVENT_TYPES, "other"), ("event_action", ALLOWED_ACTIONS, "new" if repaired["relevant"] else "irrelevant"),
        ("direction", ALLOWED_DIRECTIONS, "neutral"), ("scope", ALLOWED_SCOPES, "ticker" if repaired["relevant"] else "irrelevant"),
        ("materiality", ALLOWED_LEVELS, "none"), ("novelty", ALLOWED_LEVELS, "none"), ("confidence", ALLOWED_LEVELS, "none"),
    ]:
        value = normalise_enum(repaired.get(field, default))
        repaired[field] = VALUE_ALIASES.get(field, {}).get(value, value)
        if repaired[field] not in allowed:
            repaired[field] = default

    channels = repaired.get("transmission_channels", ["none"])
    channels = channels if isinstance(channels, list) else [channels]
    repaired["transmission_channels"] = list(dict.fromkeys(
        VALUE_ALIASES["transmission_channels"].get(normalise_enum(channel), normalise_enum(channel))
        for channel in channels if VALUE_ALIASES["transmission_channels"].get(normalise_enum(channel), normalise_enum(channel)) in ALLOWED_CHANNELS
    )) or ["none"]

    if not repaired["relevant"] or repaired["event_action"] == "irrelevant":
        repaired.update(relevant=False, matched_event_id=None, event_action="irrelevant", event_name="No material event", event_type="other", direction="neutral", scope="irrelevant", transmission_channels=["none"], materiality="none", novelty="none", confidence="none")
    elif repaired["event_action"] == "new" or repaired.get("matched_event_id") not in candidate_ids:
        repaired["event_action"], repaired["matched_event_id"] = "new", None

    if not isinstance(repaired.get("event_name"), str) or not repaired["event_name"].strip():
        repaired["event_name"] = str(article.get("title") or "Unnamed event")[:50]
    else:
        repaired["event_name"] = repaired["event_name"].strip()[:50]
    return repaired


def json_objects(text: str) -> list[dict]:
    decoder, objects = json.JSONDecoder(), []
    for index, character in enumerate(text):
        if character == "{":
            try:
                value, _ = decoder.raw_decode(text[index:])
                if isinstance(value, dict): objects.append(value)
            except json.JSONDecodeError:
                pass
    return objects


def extract_event_json(raw_text: str, context_packet: dict) -> ArticleEventAssessment:
    candidates = [item for item in json_objects(raw_text) if "r" in item or "relevant" in item or "a" in item or "event_action" in item]
    if not candidates:
        raise ValueError("No assessment JSON object found in the model response")
    return model_validate_json_compat(ArticleEventAssessment, json.dumps(sanitise_assessment_payload(candidates[-1], context_packet)))

In [33]:
class ArticleEventAssessmentBatch(BaseModel):
    assessments: list[ArticleEventAssessment] = Field(description="One assessment per context packet.")


def build_batch_prompt(context_packets: list[dict]) -> str:
    return f'''{COMPACT_RULES}
Return one compact object per packet in {{"items":[...]}}. Use the same keys and aliases as the single-article prompt.
P={json.dumps(context_packets, separators=(",", ":"), default=str)}'''


def grok_generate_json(prompt: str, response_schema: dict | None = None) -> str:
    if not ALLOW_LIVE_API_CALLS:
        raise RuntimeError("Live calls are blocked. Set ALLOW_LIVE_API_CALLS=True only when ready to spend credits.")
    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=[
            {"role": "system", "content": "Output only minified JSON. Never output reasoning."},
            {"role": "user", "content": prompt},
        ],
        temperature=0,
        max_completion_tokens=MAX_COMPLETION_TOKENS,
        response_format={"type": "json_object"},
    )
    return response.choices[0].message.content or ""


def generate_model_response(context_packet: dict) -> str:
    return grok_generate_json(build_prompt(context_packet))


def generate_model_responses(context_packets: list[dict]) -> list[str]:
    if len(context_packets) == 1:
        return [generate_model_response(context_packets[0])]
    payload = json.loads(grok_generate_json(build_batch_prompt(context_packets)))
    items = payload.get("items", payload.get("assessments", []))
    if len(items) != len(context_packets):
        raise ValueError(f"Grok batch returned {len(items)} assessments for {len(context_packets)} articles")
    return [json.dumps(sanitise_assessment_payload(item, packet)) for item, packet in zip(items, context_packets)]

## 6. Cheap mock mode for understanding the pipeline

Mock mode is deliberately simplistic and must **not** be used as research
output. It lets you inspect event tables, predicate grounding, and LTN reasoning
before spending GPU time.

Set `USE_MOCK_LLM = False` and `ALLOW_LIVE_API_CALLS = True` only when you intend to call Grok.

In [34]:
def mock_assessment(article: pd.Series) -> ArticleEventAssessment:
    score = float(article.av_ticker_sentiment_score)
    direction = "risk_on" if score > 0.15 else "risk_off" if score < -0.15 else "neutral"
    magnitude = abs(score)
    materiality = "high" if magnitude > 0.35 else "medium" if magnitude > 0.15 else "low"
    title_lower = article.title.lower()

    if any(term in title_lower for term in ["supply", "chip", "manufactur"]):
        event_type = "supply_chain"
        channels = ["supply_chain"]
    elif any(term in title_lower for term in ["demand", "sales", "consumer"]):
        event_type = "demand_change"
        channels = ["demand_weakening"] if direction == "risk_off" else ["revenue_growth"]
    elif any(term in title_lower for term in ["market", "stock", "value"]):
        event_type = "market_sentiment"
        channels = ["market_uncertainty"]
    else:
        event_type = "other"
        channels = ["none"]

    return ArticleEventAssessment(
        article_uid=article.article_uid,
        ticker=article.ticker,
        relevant=True,
        matched_event_id=None,
        event_action="new",
        event_name=article.title[:100],
        event_type=event_type,
        direction=direction,
        scope="ticker",
        transmission_channels=channels,
        materiality=materiality,
        novelty="medium",
        confidence="medium"
    )

## 7. Process articles chronologically and build the graph

With `GROK_BATCH_SIZE = 1`, each call receives only events and assessments
created before the current article timestamp. Larger batches reduce request count
for big runs, but articles inside the same batch share the same pre-batch memory
state.


In [35]:
memory = TemporalEventMemory()
assessment_records = []
raw_response_records = []
context_records = []

articles = list(article_stream.iterrows())
batch_size = max(1, GROK_BATCH_SIZE)

for start in range(0, len(articles), batch_size):
    batch = articles[start:start + batch_size]
    context_packets = []
    batch_articles = []

    for _, article in batch:
        context_packet = memory.context_packet(article)
        context_packets.append(context_packet)
        context_records.append(context_packet)
        batch_articles.append(article)

    try:
        if USE_MOCK_LLM:
            raw_texts = ["MOCK MODE" for _ in batch_articles]
            assessments = [mock_assessment(article) for article in batch_articles]
        else:
            raw_texts = generate_model_responses(context_packets)
            assessments = [
                extract_event_json(raw_text, context_packet)
                for raw_text, context_packet in zip(raw_texts, context_packets)
            ]
    except Exception as exc:
        raw_text = raw_texts[0] if "raw_texts" in locals() and raw_texts else ""
        for article in batch_articles:
            raw_response_records.append({
                "article_uid": article.article_uid,
                "time_published": pd.Timestamp(article.time_published).isoformat(),
                "raw_response": raw_text,
                "error": f"{type(exc).__name__}: {exc}",
            })
            print(f"Skipped {article.article_uid}: {type(exc).__name__}: {exc}")
    else:
        for article, assessment, raw_text in zip(batch_articles, assessments, raw_texts):
            raw_response_records.append({
                "article_uid": article.article_uid,
                "time_published": pd.Timestamp(article.time_published).isoformat(),
                "raw_response": raw_text,
                "error": None,
            })

            event_id = memory.apply(assessment, pd.Timestamp(article.time_published))

            assessment_records.append({
                **model_dump_compat(assessment),
                "resolved_event_id": event_id,
                "time_published": pd.Timestamp(article.time_published).isoformat(),

                # Raw article fields for manual inspection
                "article_title": article.title,
                "article_source": article.source,
                "alphavantage_ticker_relevance": article.ticker_relevance,
                "alphavantage_ticker_sentiment_score": article.av_ticker_sentiment_score,
                "alphavantage_ticker_sentiment_label": article.av_ticker_sentiment_label,
            })

    # Save an auditable checkpoint after every batch, including failures.
    pd.DataFrame(assessment_records).to_json(
        OUTPUT_DIR / "article_assessments.jsonl", orient="records", lines=True,
    )
    pd.DataFrame(raw_response_records).to_json(
        OUTPUT_DIR / "raw_model_responses.jsonl", orient="records", lines=True,
    )
    pd.DataFrame(context_records).to_json(
        OUTPUT_DIR / "context_packets.jsonl",
        orient="records",
        lines=True,
        default_handler=str,
    )
    events_df, states_df, evidence_df = memory.tables()
    events_df.to_parquet(OUTPUT_DIR / "events_current.parquet", index=False)
    states_df.to_parquet(OUTPUT_DIR / "event_state_history.parquet", index=False)
    evidence_df.to_parquet(OUTPUT_DIR / "event_evidence.parquet", index=False)

    if not USE_MOCK_LLM and GROK_SLEEP_SECONDS > 0 and start + batch_size < len(articles):
        time.sleep(GROK_SLEEP_SECONDS)

assessments_df = pd.DataFrame(assessment_records)
events_df, states_df, evidence_df = memory.tables()
print(f"Processed {len(assessment_records)} of {len(article_stream)} articles.")
display(assessments_df.head(10))


c:\Users\kiera\miniconda3\envs\diss-notebook\lib\site-packages\msal\oauth2cli\oauth2.py:453: UserWarning: response_mode='form_post' is recommended for better security. See https://www.rfc-editor.org/rfc/rfc9700.html#section-4.3.1
  warnings.warn(


Processed 1 of 1 articles.


,article_uid,ticker,relevant,matched_event_id,event_action,event_name,event_type,direction,scope,transmission_channels,materiality,novelty,confidence,resolved_event_id,time_published,article_title,article_source,alphavantage_ticker_relevance,alphavantage_ticker_sentiment_score,alphavantage_ticker_sentiment_label
0,cc943125f2c299c334e9484c073748f4,AAPL,False,None,irrelevant,No material event,other,neutral,irrelevant,[none],none,none,none,None,2023-01-02T04:52:00,What is TSMC?,Tech Monitor,0.718922,0.307854,Somewhat-Bullish


In [36]:
len(assessments_df)

1

In [37]:
display(assessments_df)

,article_uid,ticker,relevant,matched_event_id,event_action,event_name,event_type,direction,scope,transmission_channels,materiality,novelty,confidence,resolved_event_id,time_published,article_title,article_source,alphavantage_ticker_relevance,alphavantage_ticker_sentiment_score,alphavantage_ticker_sentiment_label
0,cc943125f2c299c334e9484c073748f4,AAPL,False,None,irrelevant,No material event,other,neutral,irrelevant,[none],none,none,none,None,2023-01-02T04:52:00,What is TSMC?,Tech Monitor,0.718922,0.307854,Somewhat-Bullish


In [38]:
response_log_df = pd.read_json(
    OUTPUT_DIR / "raw_model_responses.jsonl",
    lines=True,
)

failed = response_log_df[response_log_df["error"].notna()]

for _, row in failed.iterrows():
    print("\nARTICLE:", row["article_uid"])
    print("ERROR:", row["error"])
    print("RAW RESPONSE:")
    print(row["raw_response"][:3000])

In [39]:
response_log_df = pd.DataFrame(
    raw_response_records,
    columns=["article_uid", "time_published", "raw_response", "error"],
)
failed_df = response_log_df.loc[response_log_df["error"].notna()].copy()

print(f"Successful responses: {len(response_log_df) - len(failed_df)}")
print(f"Failed responses: {len(failed_df)}")
if not failed_df.empty:
    display(failed_df[["article_uid", "error", "raw_response"]])


Successful responses: 1
Failed responses: 0


In [40]:
print("Current events")
display(events_df)

print("Event state history")
display(states_df.head(10))

print("Event evidence")
display(evidence_df.head(10))

Current events


""


Event state history


""


Event evidence


""


## 8. Ground event attributes into stable predicates

The event graph preserves specific facts. Predicates express reusable logical
statements.

Example:

```text
Specific event:
    "Taiwan semiconductor conflict risk"

Stable predicates:
    GeopoliticalRisk(AAPL, date)
    SupplyChainRisk(AAPL, date)
    UnresolvedMaterialRisk(AAPL, date)
```

The grounding functions below are documented and deterministic. Therefore,
`NewsRiskOff(AAPL, date) = 0.72` is not an arbitrary dictionary value: it is the
truth degree returned by a defined function applied to validated evidence.

In [ ]:
def fuzzy_or(values) -> float:
    result = 0.0
    for value in values:
        value = max(0.0, min(1.0, float(value)))
        result = result + value - result * value
    return float(result)


def latest_visible_events(states: pd.DataFrame, as_of: pd.Timestamp) -> pd.DataFrame:
    if states.empty:
        return pd.DataFrame()

    history = states.copy()
    history["valid_from"] = pd.to_datetime(history["valid_from"])

    visible = (
        history.loc[history["valid_from"] <= as_of]
        .sort_values("valid_from")
        .groupby("event_id", as_index=False)
        .tail(1)
    )

    return visible.loc[visible["status"] != "resolved"].reset_index(drop=True)


def event_truths(event: pd.Series, as_of: pd.Timestamp) -> dict[str, float]:
    materiality = LEVEL_VALUE[event.materiality]
    confidence = LEVEL_VALUE[event.confidence]

    age_days = max(0, (as_of - pd.Timestamp(event.valid_from)).days)
    decay = 0.5 ** (age_days / 30.0)

    strength = materiality * confidence * decay

    return {
        "event_risk_on": strength if event.direction == "risk_on" else 0.0,
        "event_risk_off": strength if event.direction == "risk_off" else 0.0,
        "event_material": materiality * decay,
        "unresolved_material_risk": strength
        if event.direction == "risk_off"
        else 0.0,
        "geopolitical_risk": strength
        if event.event_type == "geopolitical_conflict"
        else 0.0,
        "regulatory_risk": strength
        if event.event_type == "regulatory_or_legal"
        else 0.0,
        "supply_chain_risk": strength
        if "supply_chain" in event.transmission_channels
        else 0.0,
        "demand_weakening": strength
        if "demand_weakening" in event.transmission_channels
        else 0.0,
        "input_cost_pressure": strength
        if "input_cost_pressure" in event.transmission_channels
        else 0.0,
        "operational_disruption": strength
        if "operational_disruption" in event.transmission_channels
        else 0.0,
        "market_uncertainty": strength
        if "market_uncertainty" in event.transmission_channels
        else 0.0,
    }

In [ ]:
if assessments_df.empty:
    raise ValueError(
        "No valid article assessments were produced. "
        "Inspect raw_model_responses.jsonl for parsing or validation errors."
    )

dates = sorted(
    pd.to_datetime(assessments_df["time_published"])
    .dt.normalize()
    .unique()
)

daily_rows = []

for date in dates:
    as_of = pd.Timestamp(date) + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
    visible_events = latest_visible_events(states_df, as_of)

    row = {"ticker": TICKER, "date": pd.Timestamp(date)}

    event_rows = [
        event_truths(event, as_of)
        for _, event in visible_events.iterrows()
    ]

    predicate_names = sorted({name for event in event_rows for name in event})

    for predicate in predicate_names:
        row[predicate] = fuzzy_or(
            event.get(predicate, 0.0)
            for event in event_rows
        )

    row["active_event_count"] = len(visible_events)

    if visible_events.empty:
        row["risk_off_event_count"] = 0
        row["risk_on_event_count"] = 0
        row["conflicting_events"] = 0.0
    else:
        row["risk_off_event_count"] = int((visible_events["direction"] == "risk_off").sum())
        row["risk_on_event_count"] = int((visible_events["direction"] == "risk_on").sum())
        row["conflicting_events"] = min(
            row.get("event_risk_on", 0.0),
            row.get("event_risk_off", 0.0),
        )

    daily_rows.append(row)

daily_predicates = (
    pd.DataFrame(daily_rows)
    .fillna(0.0)
    .sort_values("date")
    .reset_index(drop=True)
)

display(daily_predicates)

In [ ]:
if response_log_df.empty:
    print("No model responses were recorded.")
else:
    print(response_log_df["error"].value_counts(dropna=False))
    display(response_log_df[["article_uid", "error", "raw_response"]].head())


## 9. LTN reasoning over the grounded predicates

LTN does not magically discover the meaning of predicates. The previous cells
performed their grounding.

Here, LTN fuzzy operators evaluate explicit reusable rules:

```text
NewsRiskOff Ã¢Ë†Â§ NewsMaterial Ã¢Ë†Â§ UnresolvedMaterialRisk Ã¢â€ â€™ BearSupport
NewsRiskOn Ã¢Ë†Â§ NewsMaterial Ã¢Ë†Â§ Ã‚Â¬UnresolvedMaterialRisk Ã¢â€ â€™ BullSupport
ConflictingNews Ã¢Ë†Â¨ (NewsMaterial Ã¢Ë†Â§ Ã‚Â¬NewsRiskOn Ã¢Ë†Â§ Ã‚Â¬NewsRiskOff) Ã¢â€ â€™ NeutralSupport
```

This is a transparent inference demonstration. Later, the regime predicates or
rule weights can be made learnable using labelled regime data and LTN
satisfiability as a training objective.

In [ ]:
try:
    import ltn

    And = ltn.fuzzy_ops.AndProd()
    Or = ltn.fuzzy_ops.OrProbSum()
    Not = ltn.fuzzy_ops.NotStandard()
    Implies = ltn.fuzzy_ops.ImpliesReichenbach()
    LTN_BACKEND = "LTNtorch"
except ImportError:
    # Transparent fallback for inspecting the notebook before LTNtorch is installed.
    And = lambda x, y: x * y
    Or = lambda x, y: x + y - x * y
    Not = lambda x: 1.0 - x
    Implies = lambda x, y: 1.0 - x + x * y
    LTN_BACKEND = "transparent PyTorch fallback"

print("Logic backend:", LTN_BACKEND)


def tensor_column(frame: pd.DataFrame, name: str) -> torch.Tensor:
    if name not in frame:
        return torch.zeros(len(frame), dtype=torch.float32)
    return torch.tensor(frame[name].fillna(0.0).to_numpy(), dtype=torch.float32)


risk_off = tensor_column(daily_predicates, "news_risk_off")
risk_on = tensor_column(daily_predicates, "news_risk_on")
material = tensor_column(daily_predicates, "news_material")
unresolved = tensor_column(daily_predicates, "unresolved_material_risk")
conflicting = tensor_column(daily_predicates, "conflicting_news")
escalated = tensor_column(daily_predicates, "existing_event_escalated")
geopolitical = tensor_column(daily_predicates, "geopolitical_risk")
supply_chain = tensor_column(daily_predicates, "supply_chain_risk")

bear_rule_1 = And(And(risk_off, material), unresolved)
bear_rule_2 = And(And(escalated, geopolitical), supply_chain)
bear_support = Or(bear_rule_1, bear_rule_2)

bull_support = And(And(risk_on, material), Not(unresolved))
weak_direction = And(Not(risk_on), Not(risk_off))
neutral_support = Or(conflicting, And(material, weak_direction))

scores = torch.stack([bear_support, neutral_support, bull_support], dim=1)
scores = scores / scores.sum(dim=1, keepdim=True).clamp_min(1e-8)

ltn_results = daily_predicates[["ticker", "date"]].copy()
ltn_results[["bear_support", "neutral_support", "bull_support"]] = scores.detach().numpy()
ltn_results["detected_regime"] = [
    ["bear_drawdown", "sideways_neutral", "bull_rally"][index]
    for index in scores.argmax(dim=1).tolist()
]
display(ltn_results)

In [ ]:
# Formula-satisfaction diagnostics illustrate how LTN evaluates implications.
formula_satisfaction = pd.DataFrame(
    {
        "date": daily_predicates["date"],
        "risk_off_material_unresolved_implies_bear": Implies(
            bear_rule_1, scores[:, 0]
        ).detach().numpy(),
        "risk_on_material_implies_bull": Implies(
            And(risk_on, material), scores[:, 2]
        ).detach().numpy(),
        "conflicting_news_implies_neutral": Implies(
            conflicting, scores[:, 1]
        ).detach().numpy(),
    }
)
display(formula_satisfaction)

## 10. Inspect one prediction from evidence to rule

This final view connects:

```text
source article
Ã¢â€ â€™ article assessment
Ã¢â€ â€™ event/evidence record
Ã¢â€ â€™ daily predicate truth values
Ã¢â€ â€™ activated LTN rule
Ã¢â€ â€™ regime support
```

In [ ]:
if not daily_predicates.empty:
    selected_date = daily_predicates.iloc[-1]["date"]
    print("Selected date:", selected_date)

    print("\nArticles and assessments")
    display(
        assessments_df.loc[
            pd.to_datetime(assessments_df["time_published"]).dt.normalize() == selected_date
        ]
    )

    print("\nVisible event states")
    display(
        states_df.loc[
            pd.to_datetime(states_df["valid_from"])
            <= pd.Timestamp(selected_date) + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
        ]
    )

    print("\nGrounded predicates")
    display(daily_predicates.loc[daily_predicates["date"] == selected_date])

    print("\nLTN result")
    display(ltn_results.loc[ltn_results["date"] == selected_date])

## What to inspect before scaling

Do not process the full dataset yet. First inspect:

1. Does Grok consistently distinguish new events from repeated reporting?
2. Do matched event IDs refer only to supplied candidate events?
3. Are event types and transmission channels stable across similar articles?
4. Do ordinal materiality and novelty assignments follow the rubric?
5. Do event-state histories contain only information available at each timestamp?
6. Are predicate truth values traceable to source evidence?
7. Are the LTN rules economically defensible?

After this works for a small January sample, increase `MAX_ARTICLES`, evaluate
multiple tickers, and replace calendar-day aggregation with an exchange-calendar
function that assigns after-hours and weekend news to the correct trading date.